# Rung 42 — merged corpus, checkpoint sweep on the HELD-OUT videos

Rung 42 trains the A2 recipe + connector on a corpus that **promotes 30 of the 38 test
videos into train** (14,415 base rows + 4,969 added = 19,384). Only **8 videos** were held
back, and they are the ONLY questions this arm may be scored on.

## 🔴 The label `OOD` does not mean what it means everywhere else in this repo

`RULES §3` reads ID/OOD off the qID prefix (`heico` = OOD = Sigmoid Resection, absent from
all training). **That premise is false for rung 42.** `RESULTS_split_42.json` promoted
**8 of the 10 heico test videos into training**, so Sigmoid Resection IS in this arm's
training set. The two heico videos left (`0023`, `0027`) are *unseen videos of a seen
procedure* — the same kind of held-out that `lapchole` has always been, not the
unseen-procedure OOD the challenge scores.

`frame.metrics` will still label them `OOD`, because the qID prefix is what it reads and
forking the canonical module to relabel one rung is exactly what `RULES §1` forbids. So the
numbers below carry the library's label and this notebook carries the correction:

| cell as printed | what it actually is for rung 42 |
|---|---|
| `*_OOD` | heico `0023` + `0027` — **2 videos**, procedure SEEN in train |
| `*_ID`  | 6 lapchole videos — procedure seen in train, as always |

⇒ **this rung cannot measure OOD generalisation locally.** That instrument was spent to buy
the +45 % data. What it CAN measure is held-out-video generalisation, and the control
answers the only question that survives: is the extra data worth it *on questions neither
model trained on*.

## Control: A2 (`21_lr_2e4_v1`), zero GPU

A2 never trained on any test video, so its archived answers on all three epochs are already
valid on these 8 videos. The control is computed by RESTRICTING those answers to the same
1,283 qIDs — same code path, no re-inference, no GPU-swap drift
([[archived-results-not-bit-reproducible]] does not bite: nothing is re-generated).

## Selection axis, declared before the numbers

`RULES §6` selects by `acc_OOD` per epoch. That axis does not exist here (see above), so the
declared axis is **`bucket_mean` over the held-out set** — all four populated buckets,
ID and OOD weighted equally, which is what the challenge's final Copeland ranking does
(`RULES §4c`). The heico half is reported beside it and is **never** called OOD.

**The memorisation read, also declared up front:** 5 epochs to `token_acc` 0.980 is where
memorisation would show. Its signature is *held-out accuracy that rises and then falls* as
epochs increase. If that shape appears it is reported as memorisation, and the selected
checkpoint is the peak — never the last epoch (`RULES §6`).


In [ ]:
# --- bootstrap -------------------------------------------------------------------
import glob, json, logging, os, shutil, subprocess, sys, time
from pathlib import Path
import pandas as pd

# `swift` is shelled out to for the merge. A papermill kernel does NOT inherit the env's
# bin/ on PATH, and it must be THIS interpreter's bin (rung 39's scar).
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
for p in (REPO / "src", REPO / "vendor" / "orena-focus" / "src"):
    if p.is_dir():
        sys.path.insert(0, str(p))

# 🔴 HF_HOME before the offline flags mean anything, and before any HF import.
# MEASURED 2026-08-15, and it is NOT what rung 39's notebook says: the judge
# (Qwen/Qwen3-4B, 7.6 GB) is cached under /workspace/.cache/huggingface. /workspace/hf_cache
# holds the 27B/32B/3.5-4B and NO judge — pointing here cost one pod restart, caught by the
# judge gate two cells below rather than 45 minutes into a merged eval.
os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

from frame import ledger, metrics
from frame.config import BaselineConfig
from frame.data import load_frame_items
from frame.run import run_baseline
print("swift on PATH:", (Path(_envbin) / "swift").exists(), "| repo:", REPO, "| exp:", EXP)

In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects BELOW this cell) ------------
SMOKE       = True      # True -> 40 questions on ONE checkpoint. Full: -p SMOKE False
RUN         = "42_merged_v1"
REPO_ROOT   = "/workspace/repo_rodri"
DATA_ROOT   = "/workspace/orena-data"
CKPT_VERSION = "v0-20260814-163049"
CKPT_STEPS  = [1212, 2424, 3636, 4848, 6060]     # one per epoch, 1..5
CONTROL_RUN = "/workspace/repo_rodri/experiments/21-recipe-sweep/runs/21_lr_2e4_v1"
CONTROL_EPOCH_DIRS = ["ep1_full", "ep2_full", "ep3_full"]
KEEP_MERGED = False     # a merged checkpoint is ~17 GB and the volume has ~60 GB of headroom
N_BOOT      = 4000

In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) ----
RUN_DIR = EXP / "runs" / RUN
CKPT_ROOT = RUN_DIR / "ckpt" / CKPT_VERSION
SPLIT_JSON = EXP / "RESULTS_split_42.json"

# Resolve the data root by LOOKING. An eval on an empty data root is the classic silent zero.
DATA_ROOT = next(
    (d for d in (Path(DATA_ROOT), REPO / "external_data" / "orena-data")
     if (d / "heico" / "data" / "frame" / "test.parquet").exists()), None)
assert DATA_ROOT is not None, "no frame/test.parquet found — pull the QA parquets"

split42 = json.loads(SPLIT_JSON.read_text(encoding="utf-8"))
HELD = {tuple(v.split("/", 1)) for v in split42["videos_held_out"]}
PROMOTED = {tuple(v.split("/", 1)) for v in split42["videos_promoted"]}

# ── GATE: the held-out set is disjoint from what was promoted into train, and together
# they are the whole test set. If a video were on both sides every number below is leakage.
assert not (HELD & PROMOTED), f"LEAK: {HELD & PROMOTED} is both held out and promoted"
_probe = load_frame_items(BaselineConfig(data_root=DATA_ROOT))
_allv = {(i.dataset, i.video_id) for i in _probe}
assert HELD <= _allv and PROMOTED <= _allv, "a split video is not in the test set"
assert (HELD | PROMOTED) == _allv, "held + promoted does not cover the test set"

HELD_ITEMS = [i for i in _probe if (i.dataset, i.video_id) in HELD]
HELD_QIDS = {i.request.qID for i in HELD_ITEMS}
N_HELD = len(HELD_ITEMS)
N_HEICO = sum(1 for i in HELD_ITEMS if i.dataset == "heico")
assert N_HELD == 1283, f"expected 1283 held-out questions, got {N_HELD}"
assert len(HELD) == 8 and N_HEICO == 800

print(f"held-out: {len(HELD)} videos / {N_HELD} questions "
      f"({N_HEICO} heico + {N_HELD - N_HEICO} lapchole)")
print(f"promoted INTO train: {len(PROMOTED)} videos "
      f"({sum(1 for d, v in PROMOTED if d == 'heico')} of them heico)")
print("\n🔴 heico videos still held out:", sorted(v for d, v in HELD if d == "heico"))
print("🔴 heico videos MOVED INTO TRAIN:", len([1 for d, v in PROMOTED if d == "heico"]),
      "-> the `OOD` label below is unseen-VIDEO, not unseen-PROCEDURE")
print(f"\neffective n for a CI: {len(HELD)} videos "
      f"(2 on the heico side — no heico CI is readable, RULES §13)")
del _probe

In [ ]:
# --- PRE-FLIGHT: the LLM judge must resolve offline, BEFORE anything expensive ------
# run_baseline loads the judge only AFTER the merge and the whole inference pass, so a
# missing cache fails ~30 minutes in with everything already paid for. RAISES (RULES §7).
from transformers import AutoTokenizer

_judge = BaselineConfig().judge_model
try:
    AutoTokenizer.from_pretrained(_judge)
except Exception as exc:
    raise AssertionError(
        f"JUDGE GATE FAILED: {_judge!r} does not resolve offline ({type(exc).__name__}). "
        f"HF_HOME={os.environ.get('HF_HOME')!r}. Fix the env — do NOT disable the offline "
        "flags, the whole deployment story is offline."
    ) from exc
print(f"OK judge gate: {_judge} resolves from {os.environ.get('HF_HOME')}")

In [ ]:
# --- GATES BEFORE THE GPU (all RAISE) ---------------------------------------------
# 🔴 rc=0 is NOT evidence. AdamW's decoupled weight decay moves every tensor at zero
# gradient, so a checkpoint diff cannot separate a real run from a no-op — only the
# grad_norm log does. ([[rc-zero-is-not-evidence]])
_log = CKPT_ROOT / "logging.jsonl"
assert _log.exists(), f"no logging.jsonl at {_log}"
_rows = [json.loads(l) for l in _log.read_text(encoding="utf-8").splitlines() if l.strip()]
_g = [r["grad_norm"] for r in _rows if isinstance(r.get("grad_norm"), (int, float))]
_l = [r["loss"] for r in _rows if isinstance(r.get("loss"), (int, float))]
_zero = sum(1 for v in _g if v == 0.0)
print(f"grad_norm: {len(_g)} logged steps, {_zero} of them exactly 0.0, "
      f"first {_g[0]:.3f} -> last {_g[-1]:.3f}")
print(f"loss: first {_l[0]:.3f} -> last {_l[-1]:.3f}")
assert _g and _zero == 0, f"{_zero} steps logged grad_norm 0.0 — this run moved no weight"
assert _l[-1] < _l[0], "loss did not fall"
_ta = [r["token_acc"] for r in _rows if isinstance(r.get("token_acc"), (int, float))]
if _ta:
    print(f"token_acc (TRAIN, not a result): first {_ta[0]:.3f} -> last {_ta[-1]:.3f}"
          f"  <- 5 epochs; the held-out curve below is what says whether this is memorisation")

# every checkpoint must be complete before a single GPU-second is spent
for step in CKPT_STEPS:
    d = CKPT_ROOT / f"checkpoint-{step}"
    assert (d / "adapter_model.safetensors").exists(), f"incomplete checkpoint: {d}"
print("checkpoints complete:", [f"checkpoint-{s}" for s in CKPT_STEPS])

# WARNS, never blocks. `du` against the configured quota — df reports the MooseFS cluster
# at ~314 TB while the volume carries an invisible ~640 GB wall, and a run died mid-merge.
_used = int(subprocess.run(["du", "-sx", "/workspace"], capture_output=True,
                           text=True).stdout.split()[0]) / 1048576
print(f"disk: {_used:.0f} GB used of the ~640 GB empirical wall -> {640 - _used:.0f} GB free "
      f"(one merge is ~17 GB and is reclaimed immediately)")

In [ ]:
# --- the CONTROL, computed FIRST and with zero GPU ---------------------------------
# A2 (`21_lr_2e4_v1`) never trained on any test video, so its archived answers are valid on
# these 8 videos. Restricting them to the same 1,283 qIDs is the whole control: same code
# path, no re-inference, so no GPU-swap drift.
gold = ledger.gold_from_frame_parquets(DATA_ROOT)

def score_on_heldout(results_csv: Path, label: str) -> tuple[dict, pd.DataFrame]:
    """Canonical scoring of one arm restricted to the held-out questions. RULES §1."""
    res = pd.read_csv(results_csv)
    res = res[res["qID"].isin(HELD_QIDS)].copy()
    assert len(res) == N_HELD, f"{label}: {len(res)} of {N_HELD} held-out questions scored"
    missing = set(res["qID"]) - set(gold.dropna(subset=["answer"])["qID"])
    assert not missing, f"{label}: {len(missing)} qIDs without gold — every margin inflated"
    metrics.assert_no_dup_qid(res)
    metrics.assert_ood_from_qid(res)
    metrics.assert_all_rows_grouped(res)
    # n_boot=0 is NOT a supported value: _hier_bootstrap builds an empty array and
    # np.percentile then raises IndexError. Measured 2026-08-15; a smoke uses a cheap 200.
    strat = metrics.stratified_report(res, gold=gold, n_boot=200 if SMOKE else 1000)
    metrics.assert_floors_vs_eval_set(strat)
    return strat, res

def cells(report: dict) -> dict:
    bb = pd.DataFrame(report["by_bucket"])
    out = {}
    for dist in ("ID", "OOD"):
        d = bb[bb.distribution == dist].set_index("capability_group")
        for name in ("aggregation", "object_recognition"):
            out[f"{name}_{dist}"] = float(d.loc[name, "accuracy"]) if name in d.index else float("nan")
    out["bucket_mean"] = float(report["bucket_mean"])
    out["acc_ID"] = float(report["acc_ID"]); out["acc_OOD"] = float(report["acc_OOD"])
    out["margin_ID"] = float(report["margin_ID"]); out["margin_OOD"] = float(report["margin_OOD"])
    return out

CTRL = {}
for i, epd in enumerate(CONTROL_EPOCH_DIRS, 1):
    csv = Path(CONTROL_RUN) / epd / "results.csv"
    if not csv.exists():
        print(f"⚠️  control {epd}: no results.csv — that epoch is a MISSING CONTROL (RULES §6b)")
        continue
    s, r = score_on_heldout(csv, f"A2 {epd}")
    CTRL[i] = {"strat": s, "res": r, "cells": cells(s), "dir": epd}
    print(f"A2 epoch {i} ({epd}) on the 8 held-out videos: "
          f"bucket_mean {s['bucket_mean']:.4f}  acc_ID {s['acc_ID']:.4f}  "
          f"acc_heico {s['acc_OOD']:.4f}")
assert CTRL, "no control epoch scored — the arm would have nothing to be compared against"

In [ ]:
# --- the SWEEP: merge -> eval on the held-out videos -> reclaim the 17 GB -----------
# One checkpoint at a time. The volume has ~60 GB of headroom and five merges are ~85 GB,
# so a merge that is not reclaimed immediately is the run that dies at checkpoint 4.
VIDEO_FILTER = HELD          # run_baseline restricts to exactly these (dataset, video_id)
MERGED_ROOT = RUN_DIR / "merged"
STEPS = CKPT_STEPS[:1] if SMOKE else CKPT_STEPS

def merged_model_dir(out_dir: Path) -> Path:
    """swift writes either INTO out_dir or into a `*-merged` child. Find the config.json."""
    if (out_dir / "config.json").exists():
        return out_dir
    hits = sorted(out_dir.glob("*/config.json"))
    assert hits, f"no config.json under {out_dir} — the merge produced nothing loadable"
    return hits[0].parent

def merge(step: int) -> Path:
    ck = CKPT_ROOT / f"checkpoint-{step}"
    out = MERGED_ROOT / f"checkpoint-{step}"
    if out.is_dir() and any(out.iterdir()):
        print(f"      merged already present -> {out}")
        return merged_model_dir(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    p = subprocess.run(["swift", "export", "--adapters", str(ck), "--merge_lora", "true",
                        "--output_dir", str(out)], capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-3000:]); print(p.stderr[-3000:])
        raise RuntimeError(f"swift export failed rc={p.returncode} for checkpoint-{step}")
    print(f"      merged checkpoint-{step} in {time.perf_counter() - t0:.0f}s")
    return merged_model_dir(out)

ARM = {}
for idx, step in enumerate(STEPS, 1):
    epoch = CKPT_STEPS.index(step) + 1
    tag = f"ep{epoch}_smoke" if SMOKE else f"ep{epoch}_full"
    out_csv = RUN_DIR / tag / "results.csv"
    if out_csv.exists():
        print(f"=== epoch {epoch} (checkpoint-{step}) — answers already on disk, reusing")
    else:
        print(f"=== epoch {epoch} (checkpoint-{step}) — merge + eval")
        model = merge(step)
        t0 = time.perf_counter()
        try:
            cfg_eval = BaselineConfig(
                data_root=DATA_ROOT, model_path=model, out_dir=RUN_DIR, run_name=tag,
                max_pixels=1280 * 720, n_eval=40 if SMOKE else None,
            )
            # The inference path stays rung 06's EXACTLY: this rung's variable is the
            # training corpus. A post-processor or a second sample here would be a second
            # variable and the delta would stop being attributable.
            assert cfg_eval.answer_postprocess is None and cfg_eval.n_samples == 1
            assert cfg_eval.enhance is None and cfg_eval.aux_view is None
            run_baseline(cfg_eval, video_filter=VIDEO_FILTER)
            print(f"      eval done in {(time.perf_counter() - t0) / 60:.1f} min")
        finally:
            m = MERGED_ROOT / f"checkpoint-{step}"
            if not KEEP_MERGED and m.is_dir():
                shutil.rmtree(m, ignore_errors=True)
                print(f"      reclaimed ~17 GB -> removed {m}")
    if SMOKE:
        res = pd.read_csv(out_csv)
        print(f"      SMOKE: {len(res)} rows scored (wiring only, no verdict)")
        ARM[epoch] = {"tag": tag}
        continue
    s, r = score_on_heldout(out_csv, f"rung42 epoch {epoch}")
    ARM[epoch] = {"strat": s, "res": r, "cells": cells(s), "tag": tag, "step": step}
    print(f"      bucket_mean {s['bucket_mean']:.4f}  acc_ID {s['acc_ID']:.4f}  "
          f"acc_heico {s['acc_OOD']:.4f}")

In [ ]:
# --- 🎯 the per-epoch table and the DECLARED selection -----------------------------
# Axis declared before the numbers: `bucket_mean` on the held-out set (4 buckets, ID and
# heico weighted equally — what the final Copeland ranking does, RULES §4c). `RULES §6`'s
# acc_OOD does not exist for this rung: the OOD procedure is in its training set.
if SMOKE:
    print("SMOKE — no verdict, no selection")
else:
    rows = []
    for e in sorted(ARM):
        c = ARM[e]["cells"]
        rows.append({"arm": "42_merged", "epoch": e, "ckpt": f"checkpoint-{ARM[e]['step']}",
                     **{k: round(v, 4) for k, v in c.items()}})
    for e in sorted(CTRL):
        c = CTRL[e]["cells"]
        rows.append({"arm": "A2 (control)", "epoch": e, "ckpt": CTRL[e]["dir"],
                     **{k: round(v, 4) for k, v in c.items()}})
    tbl = pd.DataFrame(rows)
    cols = ["arm", "epoch", "ckpt", "bucket_mean", "acc_ID", "acc_OOD",
            "aggregation_ID", "object_recognition_ID",
            "aggregation_OOD", "object_recognition_OOD", "margin_ID", "margin_OOD"]
    print(tbl[cols].to_string(index=False))
    print("\n⚠️  `*_OOD` / `acc_OOD` above = heico 0023 + 0027 = 2 videos whose PROCEDURE "
          "IS IN THIS ARM'S TRAINING SET. It is unseen-video, not unseen-procedure.")

    arm_curve = {e: ARM[e]["cells"]["bucket_mean"] for e in sorted(ARM)}
    best = max(arm_curve, key=arm_curve.get)       # epoch with the highest held-out score
    last = max(arm_curve.keys())                   # the last epoch trained
    rose_then_fell = best != last                  # the memorisation signature, declared above
    print(f"\nheld-out bucket_mean by epoch: "
          + "  ".join(f"ep{e} {v:.4f}" for e, v in arm_curve.items()))
    print(f"SELECTED: epoch {best} (checkpoint-{ARM[best]['step']}) — the PEAK, "
          f"never the last epoch (RULES §6)")
    if rose_then_fell:
        drop = arm_curve[last] - arm_curve[best]
        print(f"🔴 MEMORISATION SIGNATURE: held-out accuracy PEAKS at epoch {best} and moves "
              f"{drop:+.4f} by epoch {last} while train token_acc keeps rising. "
              f"The later checkpoints are fitting the 30 promoted videos, not generalising.")
    else:
        print("no rise-then-fall on the held-out curve: the last epoch is also the peak, so "
              "this sweep found no memorisation signature on `bucket_mean` "
              "(it does NOT prove there is none — read the per-cell curves above).")

In [ ]:
# --- class-balanced F1 on `fo_class` — MANDATORY before publishing (RULES §9b) -----
# `fo_class` accuracy is exact SET equality, so it is dominated by the head of a long-tailed
# class distribution and cannot see a tail collapse. Read the per_class table BESIDE the
# scalar: rung 21's +0.174 macro-F1 was 82% one `Needle` question flipping.
F1 = {}
if not SMOKE:
    f1rows = []
    for e in sorted(ARM):
        preds = metrics.predictions_frame(RUN_DIR / ARM[e]["tag"])
        preds = preds[preds["qID"].isin(HELD_QIDS)]
        F1[e] = metrics.class_f1_report(preds, gold, results_df=ARM[e]["res"], n_boot=2000)
        for cell in ("pooled", "ID", "OOD"):
            b = F1[e][cell]
            f1rows.append({"arm": "42_merged", "epoch": e, "cell": cell, "n": b["n"],
                           "illegal": b["n_illegal"], "macro_f1": round(b["macro_f1"], 4),
                           "exact_set_acc": round(b["exact_set_acc"], 4),
                           "ci": f"[{b['ci_low']:.3f}, {b['ci_high']:.3f}]"})
    CTRL_F1 = {}
    for e in sorted(CTRL):
        preds = metrics.predictions_frame(Path(CONTROL_RUN) / CTRL[e]["dir"])
        preds = preds[preds["qID"].isin(HELD_QIDS)]
        CTRL_F1[e] = metrics.class_f1_report(preds, gold, results_df=CTRL[e]["res"], n_boot=0)
        for cell in ("pooled", "ID", "OOD"):
            b = CTRL_F1[e][cell]
            f1rows.append({"arm": "A2 (control)", "epoch": e, "cell": cell, "n": b["n"],
                           "illegal": b["n_illegal"], "macro_f1": round(b["macro_f1"], 4),
                           "exact_set_acc": round(b["exact_set_acc"], 4), "ci": ""})
    f1_df = pd.DataFrame(f1rows)
    print(f1_df.to_string(index=False))

    _sel = max(ARM, key=lambda e: ARM[e]["cells"]["bucket_mean"])
    _per = pd.DataFrame(F1[_sel]["ID"]["per_class"]).T
    print(f"\n--- per class, ID, SELECTED epoch {_sel} (the tail is the point) ---")
    print("(no fo_class x ID rows)" if _per.empty else
          _per[["n_gold", "recall", "precision", "f1"]]
          .sort_values("n_gold", ascending=False).round(3).to_string())
    # 🔴 an illegal class token does not score 0 — FOType.from_name() RAISES (RULES §8b)
    _ill = f1_df[(f1_df.arm == "42_merged") & (f1_df.illegal > 0)]
    if len(_ill):
        print("\n🔴 ILLEGAL fo_class tokens emitted (RULES §8b — from_name() RAISES in "
              "verify(), so these do not merely score 0):")
        print(_ill.to_string(index=False))

In [ ]:
# --- the PAIRED, VIDEO-CLUSTERED CI vs A2 ep3 --------------------------------------
# Effective n is 8 VIDEOS, not 1,283 questions (RULES §13). An unclustered CI here would be
# ~10x too narrow and would manufacture significance. RULES §S8: only a pre-declared cell may
# GRANT a win; ANY cell may veto — so every cell is computed.
# 🔴 Two videos carry the whole heico side. No `*_OOD` CI below is readable; it is printed so
# that a VETO can still be seen, never so that a win can be claimed there.
ci_df = None
if not SMOKE and CTRL:
    ctrl_ep = max(CTRL)
    sel = max(ARM, key=lambda e: ARM[e]["cells"]["bucket_mean"])
    a = CTRL[ctrl_ep]["res"][["qID", "video", "correctness", "primary"]].rename(
        columns={"correctness": "correct_a"})
    b = ARM[sel]["res"][["qID", "correctness"]].rename(columns={"correctness": "correct_b"})
    j = a.merge(b, on="qID", how="inner")
    assert len(j) == N_HELD, f"arms not scored on the same questions ({len(j)} vs {N_HELD})"
    # RULES §2: leaf -> group ALWAYS via Capability.group (`metrics._leaf_to_group`).
    j["group"] = j["primary"].map(metrics._leaf_to_group)
    j["dist"] = j["qID"].map(lambda q: "OOD" if str(q).split("__")[0] == "heico" else "ID")

    rows = []
    for grp in sorted(j["group"].unique()) + ["ALL"]:
        for dist in ("ID", "OOD"):
            sub = j[j["dist"] == dist]
            sub = sub if grp == "ALL" else sub[sub["group"] == grp]
            rows.append({"cell": f"{grp}_{dist}",
                         **metrics.paired_delta_ci(sub, n_boot=N_BOOT, seed=42)})
    ci_df = pd.DataFrame(rows)
    ci_df["excludes_zero"] = (ci_df.ci_low > 0) | (ci_df.ci_high < 0)
    print(f"rung 42 epoch {sel} MINUS A2 epoch {ctrl_ep}, on the 8 held-out videos:")
    print(ci_df.to_string(index=False))
    vetoed = ci_df[(ci_df.excludes_zero) & (ci_df.delta < 0)]["cell"].tolist()
    won = ci_df[(ci_df.excludes_zero) & (ci_df.delta > 0)]["cell"].tolist()
    d_bm = ARM[sel]["cells"]["bucket_mean"] - CTRL[ctrl_ep]["cells"]["bucket_mean"]
    print(f"\nd(bucket_mean, held-out) = {d_bm:+.4f}")
    print(f"cells favouring rung 42 with a CI excluding zero : {won}")
    print(f"cells favouring A2 with a CI excluding zero (VETO): {vetoed}")
    print("RULES §S4: below |d| = 0.01 nothing is readable. §S1: acting needs |d| ~ 0.03, "
          "and that is a team call with the cost stated, never automatic.")
    print("⚠️  n = 8 videos. Every CI here is wide by construction and the heico half rests "
          "on TWO videos — a jackknife over videos moves acc_OOD by a median 0.024 on 38.")
    print("⚠️  GPU SWAP: the arm was generated on this pod's GPU, the control's answers were "
          "archived from another. [[archived-results-not-bit-reproducible]] measures ~0.5% of "
          "answers changing on a GPU swap — that is inside these CIs, not outside them.")

In [ ]:
# --- persist: RESULTS.csv + the ledger (full runs only) ----------------------------
if not SMOKE:
    ctrl_ep = max(CTRL)
    sel = max(ARM, key=lambda e: ARM[e]["cells"]["bucket_mean"])
    out_rows = []
    for e in sorted(ARM):
        c = ARM[e]["cells"]
        row = {
            "run": RUN, "arm": "42_merged_corpus", "epoch": e,
            "checkpoint": f"checkpoint-{ARM[e]['step']}",
            "eval_set": "8 held-out videos (1283 q)", "n_eval": N_HELD,
            "n_videos": len(HELD), "n_heico_videos": 2,
            # 🔴 the honest label: this arm TRAINED on 8 of the 10 heico test videos, so the
            # `_OOD` columns are unseen-video, not unseen-procedure.
            "ood_procedure_in_train": True,
            "baseline_run": "21_lr_2e4_v1", "baseline_epoch": ctrl_ep,
            "selected": e == sel,
            **{k: c[k] for k in ("bucket_mean", "acc_ID", "acc_OOD", "margin_ID", "margin_OOD",
                                 "aggregation_ID", "object_recognition_ID",
                                 "aggregation_OOD", "object_recognition_OOD")},
            # the key names are `macro_f1_<dist>` because that is what
            # metrics.assert_class_f1_reported looks for (RULES §9b); renaming disables it
            "macro_f1_ID": F1[e]["ID"]["macro_f1"], "macro_f1_OOD": F1[e]["OOD"]["macro_f1"],
            "d_bucket_mean_vs_A2_ep3":
                c["bucket_mean"] - CTRL[ctrl_ep]["cells"]["bucket_mean"],
        }
        metrics.assert_class_f1_reported(row, results_df=ARM[e]["res"])   # RULES §9b — RAISES
        out_rows.append(row)
    df = pd.DataFrame(out_rows)
    out = EXP / "RESULTS.csv"
    if out.exists():
        df = pd.concat([pd.read_csv(out), df], ignore_index=True)
    df.to_csv(out, index=False)
    f1_df.to_csv(EXP / "RESULTS_class_f1.csv", index=False)
    if ci_df is not None:
        ci_df.to_csv(EXP / "RESULTS_paired_ci.csv", index=False)
    (EXP / "RESULTS_heldout_eval.json").write_text(json.dumps({
        "eval_set": {"videos": sorted("/".join(v) for v in HELD), "n_questions": N_HELD,
                     "n_heico_questions": N_HEICO},
        "ood_caveat": ("8 of the 10 heico test videos were PROMOTED INTO TRAINING by this "
                       "rung's corpus merge, so `heico`/`OOD` here means unseen VIDEO of a "
                       "SEEN procedure. This arm cannot measure procedure-OOD locally."),
        "selection_axis": "bucket_mean on the 8 held-out videos (RULES §4c weighting)",
        "selected_epoch": int(sel), "selected_checkpoint": f"checkpoint-{ARM[sel]['step']}",
        "arm_curve": {f"ep{e}": ARM[e]["cells"]["bucket_mean"] for e in sorted(ARM)},
        "control_curve": {f"ep{e}": CTRL[e]["cells"]["bucket_mean"] for e in sorted(CTRL)},
    }, indent=2), encoding="utf-8")
    for e in sorted(ARM):
        ledger.register_run(RUN_DIR / ARM[e]["tag"], ARM[e]["strat"],
                            experiment="42-merged-corpus", run=f"{RUN}__{ARM[e]['tag']}",
                            model=f"42 merged-corpus epoch {e} (checkpoint-{ARM[e]['step']})",
                            date="2026-08-15")
    print("wrote", out)
else:
    print("SMOKE — nothing persisted to RESULTS.csv or the ledger")

In [ ]:
# --- eyeball: what did it actually SAY? (user rule: examples on every run) ---------
if not SMOKE:
    sel = max(ARM, key=lambda e: ARM[e]["cells"]["bucket_mean"])
    res = ARM[sel]["res"]
    preds = metrics.predictions_frame(RUN_DIR / ARM[sel]["tag"])
    fo = res[res.answer_format == "fo_class"].merge(preds, on="qID")
    print(f"--- epoch {sel}: fo_class misses (identity, not cardinality, is the usual one) ---")
    print(fo[fo.correctness == 0].head(12)[["qID", "prediction"]].to_string(index=False))

    g = gold[gold.qID.isin(HELD_QIDS)].copy()
    g["template"] = g["question"].map(metrics.template_of)
    clips = g[g.template == "How many Clips appear in this frame? Please provide a number."]
    look = clips.merge(preds, on="qID")
    if len(look):
        look["gold_n"] = look["answer"].map(metrics.read_count)
        look["pred_n"] = look["prediction"].map(metrics.read_count)
        print("\n--- predicted-vs-gold crosstab (Clips), held-out videos ---")
        print(pd.crosstab(look["gold_n"], look["pred_n"]).to_string())